In [13]:
import sys
import platform
import importlib.util

print(f"Python: {sys.version}")
print(f"Architecture: {platform.machine()}")
print(f"Operating system: {platform.platform()}")

packages = [
    'pandas',
    'numpy',
    'sklearn',
    'spacy',
    'tensorflow',
    'torch',
    'transformers',
    'fitz',  # PyMuPDF
    'streamlit'
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f'{package}: {"installed" if installed else "not installed"}')

Python: 3.12.15 (main, Oct  1 2026, 00:00:00) [GCC 14.3.1 20251022 (Red Hat 14.3.1-4)]
Architecture: s390x
Operating system: Linux-6.8.0-147-generic-s390x-with-glibc2.39
pandas: installed
numpy: installed
sklearn: installed
spacy: installed
tensorflow: installed
torch: installed
transformers: installed
fitz: not installed
streamlit: installed


In [9]:
import platform
import pandas as pd
import numpy as np
import sklearn

print("Architecture:", platform.machine())
print("Python libraries loaded successfully")
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "Carlos vive en Ciudad Juarez",
    "Ana estudia ciencia de datos"
]

X = CountVectorizer().fit_transform(texts)
print("Vectorizer test:", X.shape)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("TensorFlow test:", (tf.constant([1, 2]) + 1).numpy())
except Exception as e:
    print("TensorFlow test failed:", repr(e))

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("PyTorch test:", (torch.tensor([1, 2]) + 1).tolist())
except Exception as e:
    print("PyTorch test failed:", repr(e))
    

Architecture: s390x
Python libraries loaded successfully
pandas: 3.0.6
NumPy: 2.0.2
scikit-learn: 1.9.0
Vectorizer test: (2, 10)
TensorFlow: 2.18.1
TensorFlow test: [2 3]
PyTorch: 2.11.0+cpu
PyTorch test: [2, 3]


In [2]:
!whoami
!pwd
!uname -m
!sudo docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"
!sudo iptables -S INPUT
!ss -ltn
!git --version



ibm-user
/home/ibm-user/shared/datathon-pii-redaction/notebooks
s390x
/usr/bin/sh: line 1: sudo: command not found
/usr/bin/sh: line 1: sudo: command not found
State     Recv-Q    Send-Q       Local Address:Port        Peer Address:Port    
LISTEN    0         4096         127.0.0.53%lo:53               0.0.0.0:*       
LISTEN    0         100              127.0.0.1:35919            0.0.0.0:*       
LISTEN    0         4096            127.0.0.54:53               0.0.0.0:*       
LISTEN    0         100              127.0.0.1:52971            0.0.0.0:*       
LISTEN    0         100              127.0.0.1:56995            0.0.0.0:*       
LISTEN    0         100              127.0.0.1:52923            0.0.0.0:*       
LISTEN    0         100              127.0.0.1:56861            0.0.0.0:*       
LISTEN    0         100              127.0.0.1:51687            0.0.0.0:*       
LISTEN    0         100              127.0.0.1:48107            0.0.0.0:*       
LISTEN    0         100       

In [1]:
from pathlib import Path

notebook_path = Path.cwd() / "02_pii_baseline.ipynb"

print("Path:", notebook_path)
print("Exists:", notebook_path.exists())

if notebook_path.exists():
    print("Size:", notebook_path.stat().st_size, "bytes")
    print("Contents:", repr(notebook_path.read_text()[:200]))

Path: /home/ibm-user/shared/datathon-pii-redaction/notebooks/02_pii_baseline.ipynb
Exists: True
Size: 0 bytes
Contents: ''


In [2]:
import json
from pathlib import Path

notebook_path = Path.cwd() / "02_pii_baseline.ipynb"

if notebook_path.exists() and notebook_path.stat().st_size > 0:
    raise RuntimeError(
        "The notebook is no longer empty. Stopping to avoid overwriting it."
    )

notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# PII Detection Baseline\n",
                "\n",
                "Initial rule-based detection and redaction of synthetic "
                "Spanish-language applicant data.\n",
                "\n",
                "**Scope:** email, phone, CURP-like and RFC-like identifiers.\n",
                "\n",
                "**Limitation:** These rules are a prototype, not a validated "
                "PII detector."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "import re\n",
                "import pandas as pd\n",
                "\n",
                "PII_PATTERNS = {\n",
                "    'EMAIL': re.compile(\n",
                "        r'\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b'\n",
                "    ),\n",
                "    'PHONE': re.compile(\n",
                "        r'(?<!\\w)(?:\\+52[\\s.-]?)?(?:\\(?\\d{2,3}\\)?[\\s.-]?)?'\n",
                "        r'\\d{3,4}[\\s.-]?\\d{4}(?!\\w)'\n",
                "    ),\n",
                "    'CURP_LIKE': re.compile(\n",
                "        r'\\b[A-Z]{4}\\d{6}[HM][A-Z]{5}[A-Z0-9]\\d\\b',\n",
                "        re.IGNORECASE\n",
                "    ),\n",
                "    'RFC_LIKE': re.compile(\n",
                "        r'\\b[A-ZÑ&]{3,4}\\d{6}[A-Z0-9]{3}\\b',\n",
                "        re.IGNORECASE\n",
                "    ),\n",
                "}\n",
                "\n",
                "def detect_pii(text):\n",
                "    findings = []\n",
                "    for label, pattern in PII_PATTERNS.items():\n",
                "        for match in pattern.finditer(text):\n",
                "            findings.append({\n",
                "                'label': label,\n",
                "                'start': match.start(),\n",
                "                'end': match.end(),\n",
                "                'value': match.group(),\n",
                "            })\n",
                "    return sorted(findings, key=lambda item: item['start'])\n",
                "\n",
                "def redact_pii(text):\n",
                "    findings = detect_pii(text)\n",
                "    for item in reversed(findings):\n",
                "        replacement = f\"[{item['label']}]\"\n",
                "        text = text[:item['start']] + replacement + text[item['end']:]\n",
                "    return text\n"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Synthetic examples only; these are not real applicant records.\n",
                "sample_text = '''\n",
                "Candidata: Ana Ejemplo\n",
                "Correo: ana.ejemplo@example.com\n",
                "Teléfono: +52 656 123 4567\n",
                "CURP de prueba: GODE561231HDFXXX09\n",
                "RFC de prueba: ABCD9001011A2\n",
                "Experiencia: Analista de inventarios.\n",
                "'''\n",
                "\n",
                "findings = detect_pii(sample_text)\n",
                "display(pd.DataFrame(findings))\n",
                "print('\\nRedacted document:\\n')\n",
                "print(redact_pii(sample_text))\n"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.12.15"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

notebook_path.write_text(
    json.dumps(notebook, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Notebook recreated:", notebook_path)
print("File size:", notebook_path.stat().st_size, "bytes")

Notebook recreated: /home/ibm-user/shared/datathon-pii-redaction/notebooks/02_pii_baseline.ipynb
File size: 3195 bytes


In [3]:
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.1 MB/s  0:00:00
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for pyarrow did not run successfully.
  │ exit code: 1
  ╰─> [514 lines of output]
      Ignoring libcst: markers 'python_version >= "3.15"' don't match your environment
           ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/914.7 kB ? eta -:--:--
           ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.7/914.7 kB 24.1 MB/s  0:00:00
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
           ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/20.9 MB ? eta -:--:--
           ━━━━━━━━━━━━━━━━━

In [4]:
!pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2


In [5]:
!pip install streamlit

  Using cached streamlit-1.66.0-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached pyarrow-26.0.0.tar.gz (1.2 MB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for pyarrow did not run successfully.
  │ exit code: 1
  ╰─> [493 lines of output]
      Ignoring libcst: markers 'python_version >= "3.15"' don't match your environment
        Using cached scikit_build_core-1.1.1-py3-none-any.whl.metadata (19 kB)
        Using cached cython-3.3.0-py3-none-any.whl.metadata (28 kB)
        Using cached libcst-1.9.0.tar.gz (914 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproje

In [6]:
!pip install --upgrade pip

In [7]:
!pip install --no-deps streamlit

  Using cached streamlit-1.66.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 55.1 MB/s  0:00:00m0:00:01


In [8]:
!pip install altair click cachetools blinker packaging protobuf pydeck requests tenacity toml


  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [altair]2m5/6 [altair]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.66.0 requires itsdangerous<3,>=2.1.2, which is not installed.
streamlit 1.66.0 requires pyarrow!=25.0.0,<27,>=10.0.1, which is not installed.
streamlit 1.66.0 requires python-multipart<1,>=0.0.10, which is not installed.
streamlit 1.66.0 requires starlette<2,>=0.46.0, which is not installed.
streamlit 1.66.0 requires uvicorn<1,>=0.30.0, which is not installed.
streamlit 1.66.0 requires watchdog<7,>=2.1.5; platform_system != "Darwin", which is not installe

In [10]:
!pip install spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached murmurhash-1.0.15-cp312-cp312-linux_s390x.whl
  Using cached cymem-2.0.13-cp312-cp312-linux_s390x.whl
  Using cached preshed-3.0.13-cp312-cp312-linux_s390x.whl
  Using cached thinc-8.3.13-cp312-cp312-linux_s390x.whl
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.4-cp312-cp312-linux_s390x.whl
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.1-cp312-cp312-manylinux_2_17_s390x.manylinux2014_s390x.whl.metadata (6.5 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metad

In [14]:
!pip install pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 61.8 MB/s  0:00:01m0:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [1400 lines of output]
      [+0.0s] setup.py:184: <module>(): ### Starting.
      [+0.0s] setup.py:185: <module>(): os.getcwd()='/tmp/pip-install-y4g6wkmg/pymupdf_dc2f6098b1e7437295d2fd38caaf6e53'
      [+0.0s] setup.py:185: <module>(): platform.machine()='s390x'
      [+0.0s] setup.py:185: <module>(): platform.platform()='Linux-6.8.0-147-generic-s390x-with-glibc2.39'
      [+0.0s] setup.py:185: <module>(): platform.python_implementation()='CPython'
      [+0.0s] setup.py:185: <module>(): platform.python_version()='3.12.15'
      [+0.0s] setup.py:185: <module>(): platform.system()='Linux'
   